In [ ]:
import requests
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from scipy.stats import randint, uniform
import xgboost as xgb
import argparse
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

**Load Dataset**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Tugas Akhir Zia/ml_ready_with_hypothetical_po_beta.csv")
#ml_ready_ao_flagged
#ml_ready_2
#ml_ready_with_hypothetical_po
#ml_ready_with_hypothetical_po_beta

In [ ]:
dft = df[df["product"] == "MU-250-40Kg"]
dft["split"] = dft["split"].replace({"val":"test"})
dft.tail()

**ADF TEST**

In [ ]:
# @title
from statsmodels.tsa.stattools import adfuller, kpss, acf
from statsmodels.stats.stattools import jarque_bera
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt
import scipy.stats as stats

# Use your raw actual demand series (MU-380-40KG example)
series = dft["actual"]

# ── 1. ADF Test (stationarity) ─────────────────────────────────────────────
adf_stat, adf_p, _, _, adf_crit, _ = adfuller(series)
print("=== ADF Test (Augmented Dickey-Fuller) ===")
print(f"  ADF Statistic : {adf_stat:.4f}")
print(f"  p-value       : {adf_p:.4f}")
print(f"  Critical (5%) : {adf_crit['5%']:.4f}")
print("  →", "Stationary (reject unit root)" if adf_p < 0.05 else "Non-stationary (unit root present)")

# ── 2. KPSS Test (stationarity — opposite hypothesis to ADF) ──────────────
kpss_stat, kpss_p, _, kpss_crit = kpss(series, regression='c', nlags='auto')
print("\n=== KPSS Test ===")
print(f"  KPSS Statistic : {kpss_stat:.4f}")
print(f"  p-value        : {kpss_p:.4f}")
print(f"  Critical (5%)  : {kpss_crit['5%']:.4f}")
print("  →", "Stationary" if kpss_p > 0.05 else "Non-stationary")

# ── 3. Jarque-Bera Test (normality of distribution) ───────────────────────
jb_stat, jb_p, skew, kurtosis = jarque_bera(series)
print("\n=== Jarque-Bera Test (normality) ===")
print(f"  JB Statistic : {jb_stat:.4f}")
print(f"  p-value      : {jb_p:.4f}")
print(f"  Skewness     : {skew:.4f}")
print(f"  Kurtosis     : {kurtosis:.4f}")
print("  →", "Normal distribution" if jb_p > 0.05 else "Non-normal — supports use of non-parametric ML models")

# ── 4. Autocorrelation Check (ACF on raw series) ──────────────────────────
acf_vals = acf(series, nlags=12, fft=True)
print("\n=== Autocorrelation (ACF lags 1–12) ===")
for lag, val in enumerate(acf_vals[1:], 1):
    bar = '█' * int(abs(val) * 20)
    print(f"  lag {lag:>2}: {val:>7.4f}  {bar}")
print("  → Significant ACF confirms lag features are informative")

# ── 5. ACF / PACF Plot ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(series,  lags=12, ax=axes[0], title='ACF — Raw Demand')
plot_pacf(series, lags=12, ax=axes[1], title='PACF — Raw Demand')
plt.tight_layout()
plt.show()

**Model**

In [ ]:
FEATURE_COLS = [
  "lag_1", "lag_2", "lag_3", "lag_6", "lag_12",
    "roll_mean_3", "roll_mean_6", "roll_std_3",
    "trend_slope_12", "quarter",
    "is_lebaran_window", "is_year_end", "year_index", "fc_error_1", "fc_acc_1"
]

# Updated split: train until Sept 2025, test Oct-Dec 2025
dft["date"] = pd.to_datetime(dft["date"])
train = dft[((dft["date"] > "2021-12-01") & (dft["date"] <  "2025-03-01"))]
test  = dft[dft['date'] >= '2025-03-01']

# Keep as DataFrames to preserve feature names for LightGBM/XGBoost
X_train = train[FEATURE_COLS]
y_train = train['actual']
X_test  = test[FEATURE_COLS]
y_test  = test['actual']

dates_test = test['date'].values

print("Train:", X_train.shape, "\u2192", train['date'].min().date(), "to", train['date'].max().date())
print("Test: ", X_test.shape,  "\u2192", test['date'].min().date(),  "to", test['date'].max().date())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Calculate correlation of features with target in the training set
train_data = X_train.copy()
train_data['actual'] = y_train
correlations = train_data.corr()['actual'].drop('actual')

# 2. Calculate average of absolute positive values (absolute correlations)
abs_corrs = correlations.abs()
avg_abs_corr = abs_corrs.mean()
print(f"Average Absolute Correlation: {avg_abs_corr:.4f}")

# 3. Filter features marked more than the average
selected_by_corr = abs_corrs[abs_corrs > avg_abs_corr].index.tolist()
print(f"Selected Features (> Avg): {selected_by_corr}")

# Update FEATURE_COLS for downstream cells
FEATURE_COLS = selected_by_corr

# 4. Visualize results
correlations_sorted = correlations.sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=correlations_sorted.values, y=correlations_sorted.index, palette='coolwarm', hue=correlations_sorted.index, legend=False)
plt.axvline(x=avg_abs_corr, color='red', linestyle='--', label=f'Avg Abs Corr ({avg_abs_corr:.2f})')
plt.axvline(x=-avg_abs_corr, color='red', linestyle='--')
plt.title('Feature Correlation with Target')
plt.xlabel('Correlation Coefficient')
plt.ylabel('Features')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()

# 5. Display numerical values
display(correlations_sorted.to_frame(name='Correlation with Actual'))

In [ ]:
# Re-Fit the dataset with selected features

X_train = train[FEATURE_COLS]
y_train = train['actual']
X_test  = test[FEATURE_COLS]
y_test  = test['actual']

dates_test = test['date'].values

# Calculate correlation matrix for the selected features in the training set
corr_matrix = X_train.corr()

# Visualize the multicollinearity with a heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Multicollinearity Matrix (Selected Features)')
plt.show()

In [ ]:
def mape(y_true, y_pred):
    mask = (y_true != 0) & ~np.isnan(y_true)
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

In [ ]:
naive_test = test["lag_12"] #The naive test (or naive forecast) is a benchmark forecasting method where the next period's value is predicted to be the same as the last observed value
human_test = test["forecast"]

print("=== BASELINE ===")
print("Naive MAPE:", mape(y_test, naive_test))
print("Human MAPE:", mape(y_test, human_test))

In [ ]:
# model XGBoost
model_xgb = xgb.XGBRegressor(random_state=42)
model_xgb.fit(X_train, y_train)

pred_xgb = model_xgb.predict(X_test)
print("\n=== XGBoost Model ===")
print("Test MAPE:", mape(y_test, pred_xgb))

In [ ]:
#Model Random Forest
from sklearn.ensemble import RandomForestRegressor
model_rf = RandomForestRegressor(random_state=42)
model_rf.fit(X_train, y_train)

pred_rf = model_rf.predict(X_test)
print("\n=== Random Forest ===")
print("Test MAPE:", mape(y_test, pred_rf))

In [ ]:
#Model Prophet
from prophet import Prophet
# Prophet needs a DataFrame with columns 'ds' (date) and 'y' (target)
prophet_train = pd.DataFrame({'ds': train["date"], 'y': y_train})

model_prophet = Prophet()
model_prophet.fit(prophet_train)

future       = pd.DataFrame({'ds': dates_test})
prophet_pred = model_prophet.predict(future)['yhat'].values

print("\n=== Prophet ===")
print("Test MAPE:", mape(y_test, prophet_pred))

In [ ]:
#Model Lasso
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

# Lasso needs scaled X
scaler_lasso = StandardScaler()
X_train_sc   = scaler_lasso.fit_transform(X_train)
X_test_sc    = scaler_lasso.transform(X_test)

model_lasso = Lasso(alpha=0.1, max_iter=10000)
model_lasso.fit(X_train_sc, y_train)

pred_lasso = model_lasso.predict(X_test_sc)
print("\n=== Lasso Regression ===")
print("Test MAPE:", mape(y_test, pred_lasso))

In [ ]:
import lightgbm as lgb

# Adjusting parameters for the updated training set (45 rows)
model_lgb = lgb.LGBMRegressor(
    n_estimators=100,
    learning_rate=0.05,
    min_child_samples=5,
    min_child_weight=0.001,
    num_leaves=7,
    random_state=42,
    verbosity=-1
)

model_lgb.fit(X_train, y_train)

pred_lgb = model_lgb.predict(X_test)
print("\n=== LightGBM (Updated) ===")
print(f"Predictions size: {len(pred_lgb)}")
print("Test MAPE:", mape(y_test, pred_lgb))

In [ ]:
# Ensemble simple average
# All prediction arrays (pred_lgb, pred_rf, pred_xgb, prophet_pred, pred_lasso) now have length 3
pred_ensemble = (pred_lgb + pred_rf + pred_xgb + prophet_pred + pred_lasso) / 5

print("\n=== Ensemble (simple average) ===")
print("Test MAPE:", mape(y_test.values, pred_ensemble))

In [ ]:
import matplotlib.pyplot as plt

# Ensure we are using the latest prediction variables from the kernel
# All should have length 3 (Oct, Nov, Dec 2025)
dates_test_plot = test["date"]

plt.figure(figsize=(12,6))

plt.plot(dates_test_plot, y_test, label="Actual", marker='o', linewidth=2, color='black')
plt.plot(dates_test_plot, pred_xgb, label="XGBoost", marker='o', alpha=0.7)
plt.plot(dates_test_plot, human_test.loc[test.index].values, label="Human Forecast", linestyle='--', color='gray')
plt.plot(dates_test_plot, pred_lgb, label="LightGBM", marker='o', alpha=0.7)
plt.plot(dates_test_plot, pred_rf, label="Random Forest", marker='o', alpha=0.7)
plt.plot(dates_test_plot, prophet_pred, label="Prophet", marker='o', alpha=0.7)
plt.plot(dates_test_plot, pred_lasso, label="Lasso", marker='o', alpha=0.7)
plt.plot(dates_test_plot, pred_ensemble, label="Ensemble", marker='s', linewidth=2, color='red')

plt.title("Demand Forecast vs Actual (Oct-Dec 2025 Test Period)")
plt.xlabel("Date")
plt.ylabel("Demand")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)

plt.tight_layout()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import pandas as pd

# Collect all MAPE results
results = {
    'Model': ['Human Forecast', 'Naive (Lag 12)', 'XGBoost', 'Random Forest', 'Prophet', 'Lasso', 'LightGBM', 'Ensemble'],
    'MAPE (%)': [
        mape(y_test, human_test),
        mape(y_test, naive_test),
        mape(y_test, pred_xgb),
        mape(y_test, pred_rf),
        mape(y_test, prophet_pred),
        mape(y_test, pred_lasso),
        mape(y_test, pred_lgb),
        mape(y_test.values, pred_ensemble)
    ]
}

summary_df = pd.DataFrame(results)

# Define the threshold (Human MAPE)
human_mape = summary_df.loc[summary_df['Model'] == 'Human Forecast', 'MAPE (%)'].values[0]

# Add Flag
summary_df['Beats Human?'] = summary_df.apply(lambda row: '✅ Yes' if row['MAPE (%)'] < human_mape and row['Model'] != 'Human Forecast' else ('-' if row['Model'] == 'Human Forecast' else 'No'), axis=1)

# Sort by performance
summary_df = summary_df.sort_values(by='MAPE (%)').reset_index(drop=True)

print(f"Reference Human MAPE: {human_mape:.2f}%")
display(summary_df)

### Time-Series Cross-Validation (Lasso)
This validates the model by training on expanding windows of time to ensure the results aren't just a fluke of the specific Oct-Dec window.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Combine data for walk-forward logic
X_full_sc = np.vstack([X_train_sc, X_test_sc])
y_full = pd.concat([y_train, y_test])

# To test on the last 10 points with 10 folds, each fold tests 1 point (1 month).
# This ensures the model renews/updates after each month.
tscv = TimeSeriesSplit(n_splits=10, test_size=1)
cv_scores = []

# Execute 10 folds: Retrain after every month and predict the next
for i, (train_index, test_index) in enumerate(tscv.split(X_full_sc)):
    X_tr, X_val = X_full_sc[train_index], X_full_sc[test_index]
    y_tr, y_val = y_full.iloc[train_index], y_full.iloc[test_index]

    # Refit Lasso on the expanding window
    model_lasso.fit(X_tr, y_tr)
    preds = model_lasso.predict(X_val)

    mape_val = mape(y_val.values, preds)
    cv_scores.append(mape_val)


print(f"Lasso CV MAPE (Monthly Renewal): {np.mean(cv_scores):.2f}% (+/- {np.std(cv_scores):.2f}%)")

### XGBoost Walk-Forward CV
Applying the monthly renewal strategy to the XGBoost model.

In [ ]:
X_full = pd.concat([X_train, X_test])
y_full = pd.concat([y_train, y_test])

tscv = TimeSeriesSplit(n_splits=10, test_size=1)
xgb_cv_scores = []

for i, (train_index, test_index) in enumerate(tscv.split(X_full)):
    X_tr, X_val = X_full.iloc[train_index], X_full.iloc[test_index]
    y_tr, y_val = y_full.iloc[train_index], y_full.iloc[test_index]

    model_xgb.fit(X_tr, y_tr)
    preds = model_xgb.predict(X_val)

    m_val = mape(y_val.values, preds)
    xgb_cv_scores.append(m_val)

print(f"XGBoost CV MAPE (Monthly Renewal): {np.mean(xgb_cv_scores):.2f}% (+/- {np.std(xgb_cv_scores):.2f}%)")

### Random Forest Walk-Forward CV

In [ ]:
rf_cv_scores = []

for i, (train_index, test_index) in enumerate(tscv.split(X_full)):
    X_tr, X_val = X_full.iloc[train_index], X_full.iloc[test_index]
    y_tr, y_val = y_full.iloc[train_index], y_full.iloc[test_index]

    model_rf.fit(X_tr, y_tr)
    preds = model_rf.predict(X_val)

    m_val = mape(y_val.values, preds)
    rf_cv_scores.append(m_val)

print(f"Random Forest CV MAPE (Monthly Renewal): {np.mean(rf_cv_scores):.2f}% (+/- {np.std(rf_cv_scores):.2f}%)")

### LightGBM Walk-Forward CV

In [ ]:
lgb_cv_scores = []

for i, (train_index, test_index) in enumerate(tscv.split(X_full)):
    X_tr, X_val = X_full.iloc[train_index], X_full.iloc[test_index]
    y_tr, y_val = y_full.iloc[train_index], y_full.iloc[test_index]

    model_lgb.fit(X_tr, y_tr)
    preds = model_lgb.predict(X_val)

    m_val = mape(y_val.values, preds)
    lgb_cv_scores.append(m_val)

print(f"LightGBM CV MAPE (Monthly Renewal): {np.mean(lgb_cv_scores):.2f}% (+/- {np.std(lgb_cv_scores):.2f}%)")

### Prophet Walk-Forward CV

In [ ]:
prophet_cv_scores = []
dates_full = pd.concat([train['date'], test['date']])

for i, (train_index, test_index) in enumerate(tscv.split(X_full)):
    # Prophet needs its specific format
    p_train = pd.DataFrame({'ds': dates_full.iloc[train_index], 'y': y_full.iloc[train_index]})
    p_test_dates = pd.DataFrame({'ds': dates_full.iloc[test_index]})
    y_val = y_full.iloc[test_index]

    m_prophet = Prophet(interval_width=0.95)
    m_prophet.fit(p_train)

    forecast = m_prophet.predict(p_test_dates)
    preds = forecast['yhat'].values

    m_val = mape(y_val.values, preds)
    prophet_cv_scores.append(m_val)

print(f"Prophet CV MAPE (Monthly Renewal): {np.mean(prophet_cv_scores):.2f}% (+/- {np.std(prophet_cv_scores):.2f}%)")

### Comprehensive WFCV Comparison
Comparing the stability of all models across the 10-fold monthly renewal period.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 7))

folds = range(1, 11)
plt.plot(folds, cv_scores, marker='o', label=f'Lasso (Avg: {np.mean(cv_scores):.2f}%)', linewidth=2)
plt.plot(folds, xgb_cv_scores, marker='s', label=f'XGBoost (Avg: {np.mean(xgb_cv_scores):.2f}%)', alpha=0.8)
plt.plot(folds, rf_cv_scores, marker='^', label=f'Random Forest (Avg: {np.mean(rf_cv_scores):.2f}%)', alpha=0.8)
plt.plot(folds, lgb_cv_scores, marker='x', label=f'LightGBM (Avg: {np.mean(lgb_cv_scores):.2f}%)', alpha=0.8)
plt.plot(folds, prophet_cv_scores, marker='d', label=f'Prophet (Avg: {np.mean(prophet_cv_scores):.2f}%)', alpha=0.8)

plt.axhline(y=human_mape, color='gray', linestyle='--', label=f'Human Baseline ({human_mape:.2f}%)')

plt.title("Model Stability Comparison: MAPE per CV Fold")
plt.xlabel("Fold (Month Index in Test Set)")
plt.ylabel("MAPE (%)")
plt.xticks(folds)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Compile WFCV results into a list of dictionaries
wfcv_data = [
    {'Model': 'Lasso', 'WFCV Mean MAPE (%)': np.mean(cv_scores), 'WFCV Std (%)': np.std(cv_scores)},
    {'Model': 'XGBoost', 'WFCV Mean MAPE (%)': np.mean(xgb_cv_scores), 'WFCV Std (%)': np.std(xgb_cv_scores)},
    {'Model': 'Random Forest', 'WFCV Mean MAPE (%)': np.mean(rf_cv_scores), 'WFCV Std (%)': np.std(rf_cv_scores)},
    {'Model': 'LightGBM', 'WFCV Mean MAPE (%)': np.mean(lgb_cv_scores), 'WFCV Std (%)': np.std(lgb_cv_scores)},
    {'Model': 'Prophet', 'WFCV Mean MAPE (%)': np.mean(prophet_cv_scores), 'WFCV Std (%)': np.std(prophet_cv_scores)}
]

wfcv_summary = pd.DataFrame(wfcv_data)

# Add Human Forecast as a reference row
human_row = pd.DataFrame([{'Model': 'Human Forecast', 'WFCV Mean MAPE (%)': human_mape, 'WFCV Std (%)': 0.0}])
wfcv_summary = pd.concat([wfcv_summary, human_row], ignore_index=True)

# Add comparison flag
wfcv_summary['Beats Human?'] = wfcv_summary.apply(
    lambda row: '✅ Yes' if row['WFCV Mean MAPE (%)'] < human_mape and row['Model'] != 'Human Forecast'
    else ('-' if row['Model'] == 'Human Forecast' else 'No'),
    axis=1
)

# Sort by performance (lowest MAPE first)
wfcv_summary = wfcv_summary.sort_values(by='WFCV Mean MAPE (%)').reset_index(drop=True)

print("=== Comprehensive WFCV Summary vs Human Forecast ===")
display(wfcv_summary)